In [1]:
# CELL 1: Parsing + file sorting

import os
import re
import glob
from collections import defaultdict

import networkx as nx

# -------------------------------
# Your parser (slightly cleaned)
# -------------------------------

GATE_INST_RE = re.compile(r'^\s*(\w+)\s+(\w+)\s*\((.*)\);\s*$')

def parse_verilog_netlist(verilog_file):
    """
    Parses a gate-level Verilog netlist and constructs a directed graph.

    Returns:
        G              : DiGraph (nodes = gates + PI + PO)
        gates          : dict net_name -> (gate_name, gate_type, input_signals)
        primary_inputs : set of PI nets
        primary_outputs: set of PO nets
    """
    G = nx.DiGraph()
    gates = {}
    all_signals = set()
    output_signals = set()
    input_signals = set()

    with open(verilog_file, 'r') as f:
        lines = f.readlines()

    print(f"[INFO] {os.path.basename(verilog_file)}: {len(lines)} lines")

    for idx, line in enumerate(lines):
        if idx % 500 == 0:
            print(f"  [parse] line {idx}/{len(lines)}")

        line = line.split('//')[0].strip()  # remove comments
        if not line:
            continue

        # input / output / wire detection (simple, scalar nets)
        if line.startswith("input "):
            toks = line[len("input "):].replace(";", "")
            nets = [t.strip() for t in toks.split(",") if t.strip()]
            input_signals.update(nets)
            all_signals.update(nets)
            continue

        if line.startswith("output "):
            toks = line[len("output "):].replace(";", "")
            nets = [t.strip() for t in toks.split(",") if t.strip()]
            output_signals.update(nets)
            all_signals.update(nets)
            continue

        if line.startswith("wire "):
            toks = line[len("wire "):].replace(";", "")
            nets = [t.strip() for t in toks.split(",") if t.strip()]
            all_signals.update(nets)
            continue

        # Generic gate instantiation
        m = GATE_INST_RE.match(line)
        if m:
            cell_type, inst_name, connection_str = m.groups()

            if cell_type.lower() in {"module", "endmodule", "assign"}:
                continue

            # Signal list: first is output net, rest are input nets
            signals = [s.strip() for s in connection_str.split(",") if s.strip()]
            if len(signals) == 0:
                continue

            output_signal = signals[0]
            input_list = signals[1:]

            input_signals.update(input_list)
            all_signals.update(signals)
            output_signals.add(output_signal)

            gates[output_signal] = (inst_name, cell_type.lower(), input_list)
            G.add_node(inst_name, type=cell_type.lower())

    primary_inputs = input_signals - output_signals
    primary_outputs = output_signals - input_signals

    # Edges: producer gate / PI -> consumer gate
    for out_sig, (gate_name, gate_type, inps) in gates.items():
        for inp in inps:
            if inp in gates:
                G.add_edge(gates[inp][0], gate_name)  # gate -> gate
            elif inp in primary_inputs:
                G.add_edge(inp, gate_name)            # PI -> gate

    # Add PI/PO nodes (if not already gates)
    for pi in primary_inputs:
        if pi not in G:
            G.add_node(pi, type="input")
        else:
            G.nodes[pi]['type'] = "input"

    for po in primary_outputs:
        if po not in G:
            G.add_node(po, type="output")
        else:
            G.nodes[po]['type'] = "output"
        if po in gates:
            # gate output driving a named PO net
            G.add_edge(gates[po][0], po)

    print(f"  [done] nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
          f"PIs={len(primary_inputs)}, POs={len(primary_outputs)}")

    return G, gates, primary_inputs, primary_outputs


# ---------------------------------------
# Parse ALL netlists, sorted by file size
# ---------------------------------------

root_folder = "verilog_benchmark_circuits"  # adjust path if needed

# collect .v files and sort by size (smallest first)
verilog_files = [
    os.path.join(root_folder, f)
    for f in os.listdir(root_folder)
    if f.endswith(".v")
]

verilog_files = sorted(
    verilog_files,
    key=lambda p: os.path.getsize(p)
)

print("[INFO] Files to be parsed (smallest first):")
for p in verilog_files:
    print(f"  {os.path.basename(p):20s}  {os.path.getsize(p)} bytes")

# Dictionary to store parsed circuits
circuits = {}  # name -> dict with G, gates, primary_inputs, primary_outputs

for vf in verilog_files:
    print(f"\n[TOP] Parsing file: {os.path.basename(vf)}")
    try:
        G, gates, primary_inputs, primary_outputs = parse_verilog_netlist(vf)
        circuits[os.path.basename(vf)] = {
            "G": G,
            "gates": gates,
            "primary_inputs": primary_inputs,
            "primary_outputs": primary_outputs,
        }
    except Exception as e:
        print(f"  [ERROR] Failed on {vf}: {e}")

print(f"\n[SUMMARY] Parsed {len(circuits)} / {len(verilog_files)} circuits.")


[INFO] Files to be parsed (smallest first):
  c17.v                 529 bytes
  c432.v                7959 bytes
  c499.v                8556 bytes
  ctrl.v                12693 bytes
  c880.v                15129 bytes
  int2float.v           15348 bytes
  router.v              19829 bytes
  c1908.v               23073 bytes
  c1355.v               23906 bytes
  dec.v                 26840 bytes
  c2670.v               42021 bytes
  c3540.v               48578 bytes
  cavlc.v               48922 bytes
  Priority.v            72749 bytes
  c5315.v               80745 bytes
  i2c.v                 85411 bytes
  adder.v               94821 bytes
  c6288.v               109222 bytes
  c7552.v               117055 bytes
  bar.v                 139423 bytes
  max.v                 237623 bytes
  sin.v                 319257 bytes
  arbiter.v             984421 bytes
  voter.v               1160386 bytes
  square.v              1544660 bytes
  sqrt.v                1690731 bytes
  multiplier

In [2]:
# CELL 2: Structural manipulability M(v) computation

import math

def weisfeiler_lehman_colors(G, num_iterations=2):
    """
    Simple WL color refinement using node attribute 'type' as initial color.
    Returns: dict node -> color_string
    """
    colors = {}
    for v, data in G.nodes(data=True):
        colors[v] = f"type:{data.get('type', 'unknown')}"

    for it in range(num_iterations):
        new_colors = {}
        for v in G.nodes():
            multiset = sorted(colors[u] for u in G.neighbors(v))
            key = (colors[v], tuple(multiset))
            new_colors[v] = str(key)
        colors = new_colors
        print(f"  [WL] iteration {it+1}/{num_iterations} done")
    return colors


def compute_structural_manipulability_for_graph(
    G,
    primary_inputs,
    primary_outputs,
    alpha=0.25, beta=0.25, gamma=0.25, delta=0.25,
):
    """
    Compute structural manipulability M(v) for each node v in G.

    Components (computed on undirected graph):
      - M_path : 1 - betweenness between PI and PO (subset-based)
      - M_core : normalized k-core number
      - M_sym  : symmetry class size / |V|
      - M_cent : 1 - global betweenness

    Returns: dict node -> M(v)
    """
    G_undir = G.to_undirected()
    n_nodes = G_undir.number_of_nodes()

    if n_nodes == 0:
        return {}

    print(f"  [M] computing on undirected graph: |V|={n_nodes}, |E|={G_undir.number_of_edges()}")

    # ---------------- 1) Path-based component ----------------
    if n_nodes < 3:
        b_subset = {v: 0.0 for v in G_undir.nodes()}
    else:
        try:
            if primary_inputs and primary_outputs:
                print(f"  [M] betweenness_subset with |PI|={len(primary_inputs)}, |PO|={len(primary_outputs)}")
                b_subset = nx.betweenness_centrality_subset(
                    G_undir,
                    sources=list(primary_inputs),
                    targets=list(primary_outputs),
                    normalized=True,
                )
            else:
                print("  [M] betweenness (no explicit PI/PO)")
                b_subset = nx.betweenness_centrality(G_undir, normalized=True)
        except ZeroDivisionError:
            print("  [WARN] betweenness subset caused division by zero, setting to 0")
            b_subset = {v: 0.0 for v in G_undir.nodes()}

    M_path = {v: 1.0 - float(b_subset.get(v, 0.0)) for v in G_undir.nodes()}

    # ---------------- 2) Core decomposition ----------------
    try:
        core_num = nx.core_number(G_undir) if G_undir.number_of_edges() > 0 else {v: 0 for v in G_undir.nodes()}
    except nx.NetworkXError:
        core_num = {v: 0 for v in G_undir.nodes()}

    max_core = max(core_num.values()) if core_num else 1
    if max_core == 0:
        max_core = 1.0
    M_core = {v: core_num[v] / float(max_core) for v in G_undir.nodes()}

    # ---------------- 3) Symmetry via WL ----------------
    print("  [M] running WL colors ...")
    wl_colors = weisfeiler_lehman_colors(G_undir, num_iterations=2)
    class_sizes = defaultdict(int)
    for v, c in wl_colors.items():
        class_sizes[c] += 1
    M_sym = {v: class_sizes[wl_colors[v]] / float(n_nodes) for v in G_undir.nodes()}

    # ---------------- 4) Global betweenness ----------------
    if n_nodes < 3:
        b_global = {v: 0.0 for v in G_undir.nodes()}
    else:
        try:
            print("  [M] global betweenness ...")
            b_global = nx.betweenness_centrality(G_undir, normalized=True)
        except ZeroDivisionError:
            print("  [WARN] global betweenness division by zero, set to 0")
            b_global = {v: 0.0 for v in G_undir.nodes()}
    M_cent = {v: 1.0 - float(b_global.get(v, 0.0)) for v in G_undir.nodes()}

    # ---------------- 5) Composite M(v) ----------------
    M = {}
    for v in G_undir.nodes():
        m_val = (alpha * M_path[v] +
                 beta * M_core[v] +
                 gamma * M_sym[v] +
                 delta * M_cent[v])
        M[v] = float(max(0.0, min(1.0, m_val)))

    return M


# ---------------------------------
# Compute M(v) for all parsed circuits
# ---------------------------------

M_all = {}  # circuit_name -> dict(node -> M(v))

for cname, info in circuits.items():
    print(f"\n[TOP] Computing M(v) for circuit: {cname}")
    G = info["G"]
    primary_inputs = info["primary_inputs"]
    primary_outputs = info["primary_outputs"]

    M = compute_structural_manipulability_for_graph(G, primary_inputs, primary_outputs)
    M_all[cname] = M

    # quick sanity check: show a few nodes
    sample_nodes = list(M.keys())[:5]
    print("  [M] sample:", [(n, round(M[n], 3)) for n in sample_nodes])

print("\n[SUMMARY] Computed M(v) for", len(M_all), "circuits.")



[TOP] Computing M(v) for circuit: c17.v
  [M] computing on undirected graph: |V|=13, |E|=14
  [M] betweenness_subset with |PI|=5, |PO|=2
  [M] running WL colors ...
  [WL] iteration 1/2 done
  [WL] iteration 2/2 done
  [M] global betweenness ...
  [M] sample: [('NAND2_1', 0.714), ('NAND2_2', 0.668), ('NAND2_3', 0.644), ('NAND2_4', 0.711), ('NAND2_5', 0.686)]

[TOP] Computing M(v) for circuit: c432.v
  [M] computing on undirected graph: |V|=211, |E|=351
  [M] betweenness_subset with |PI|=36, |PO|=4
  [M] running WL colors ...
  [WL] iteration 1/2 done
  [WL] iteration 2/2 done
  [M] global betweenness ...
  [M] sample: [('NAND2_19', 0.667), ('NAND2_22', 0.667), ('NAND2_23', 0.667), ('NAND2_24', 0.671), ('NAND2_25', 0.671)]

[TOP] Computing M(v) for circuit: c499.v
  [M] computing on undirected graph: |V|=247, |E|=408
  [M] betweenness_subset with |PI|=41, |PO|=32
  [M] running WL colors ...
  [WL] iteration 1/2 done
  [WL] iteration 2/2 done
  [M] global betweenness ...
  [M] sample: [

In [5]:
# CELL 3 (UPDATED): Build PyTorch Geometric graph data with GLOBAL gate-type vocab

import numpy as np
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# ------------------------------------
# 1) Build global gate-type vocabulary
# ------------------------------------

all_gate_types = set()

for cname, info in circuits.items():
    G = info["G"]
    for n in G.nodes():
        vtype = G.nodes[n].get("type", "unknown")
        if vtype not in ("input", "output"):
            all_gate_types.add(vtype)

all_gate_types = sorted(all_gate_types)
gate_type_to_idx = {t: i for i, t in enumerate(all_gate_types)}
num_gate_types = len(all_gate_types)

print("[INFO] Global gate-type vocabulary:")
print("  count:", num_gate_types)
print("  types:", all_gate_types)


def build_pyg_data_for_circuit(G, M_full):
    """
    Build a PyG Data object for one circuit using GLOBAL gate_type_to_idx.

    Node indexing:
        - Nodes = gates + PIs + POs
        - Features:
            * one-hot gate_type (for gate nodes) w.r.t global vocab
            * is_input, is_output
            * fan_in, fan_out
        - y: M(v) for all nodes (we'll mask gates during training)
        - gate_mask: boolean mask (True for gate nodes)
    """
    node_list = list(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(node_list)}
    num_nodes = len(node_list)

    # identify gate nodes (everything that is not input/output)
    gate_nodes = [n for n in node_list if G.nodes[n].get("type") not in ("input", "output")]

    if len(gate_nodes) == 0:
        print("  [WARN] no gate nodes, skipping circuit")
        return None

    # feature layout: [one-hot gate_type (global), is_input, is_output, fan_in, fan_out]
    feat_dim = num_gate_types + 4

    X = np.zeros((num_nodes, feat_dim), dtype=np.float32)
    y = np.zeros((num_nodes, 1), dtype=np.float32)
    gate_mask = np.zeros((num_nodes,), dtype=bool)

    for v in node_list:
        idx = node_to_idx[v]
        vtype = G.nodes[v].get("type", "unknown")
        is_input = 1.0 if vtype == "input" else 0.0
        is_output = 1.0 if vtype == "output" else 0.0
        fan_in = float(G.in_degree(v))
        fan_out = float(G.out_degree(v))

        # gate-type one-hot using global vocab
        if v in gate_nodes:
            gate_mask[idx] = True
            t_idx = gate_type_to_idx.get(vtype, None)
            if t_idx is not None:
                X[idx, t_idx] = 1.0

        # add scalar features
        X[idx, num_gate_types + 0] = is_input
        X[idx, num_gate_types + 1] = is_output
        X[idx, num_gate_types + 2] = fan_in
        X[idx, num_gate_types + 3] = fan_out

        # label M(v)
        y[idx, 0] = M_full.get(v, 0.0)

    # edges: make it effectively undirected by adding both directions
    edges = []
    for u, v in G.edges():
        ui = node_to_idx[u]
        vi = node_to_idx[v]
        edges.append((ui, vi))
        edges.append((vi, ui))
    if len(edges) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(
        x=torch.from_numpy(X),
        edge_index=edge_index,
        y=torch.from_numpy(y),
    )
    data.gate_mask = torch.from_numpy(gate_mask)
    return data


# ------------------------------------
# 2) Build Data objects for all circuits
# ------------------------------------

data_list = []

for cname, info in circuits.items():
    if cname not in M_all:
        print(f"[WARN] No M(v) found for {cname}, skipping.")
        continue

    print(f"\n[TOP] Building PyG data for: {cname}")
    G = info["G"]
    M_full = M_all[cname]

    data = build_pyg_data_for_circuit(G, M_full)
    if data is None:
        continue

    data.circuit_name = cname
    data_list.append(data)

print(f"\n[SUMMARY] Built PyG graphs for {len(data_list)} circuits.")
for d in data_list[:5]:
    print(f"  {d.circuit_name}: num_nodes={d.num_nodes}, "
          f"num_edges={d.num_edges}, gate_nodes={int(d.gate_mask.sum())}, "
          f"feat_dim={d.x.size(-1)}")


[INFO] Global gate-type vocabulary:
  count: 6
  types: ['and', 'nand', 'nor', 'not', 'or', 'xor']

[TOP] Building PyG data for: c17.v

[TOP] Building PyG data for: c432.v

[TOP] Building PyG data for: c499.v

[TOP] Building PyG data for: ctrl.v

[TOP] Building PyG data for: c880.v

[TOP] Building PyG data for: int2float.v

[TOP] Building PyG data for: router.v

[TOP] Building PyG data for: c1908.v

[TOP] Building PyG data for: c1355.v

[TOP] Building PyG data for: dec.v

[TOP] Building PyG data for: c2670.v

[TOP] Building PyG data for: c3540.v

[TOP] Building PyG data for: cavlc.v

[TOP] Building PyG data for: Priority.v

[TOP] Building PyG data for: c5315.v

[TOP] Building PyG data for: i2c.v

[TOP] Building PyG data for: adder.v

[TOP] Building PyG data for: c6288.v

[TOP] Building PyG data for: c7552.v

[TOP] Building PyG data for: bar.v

[TOP] Building PyG data for: max.v

[TOP] Building PyG data for: sin.v

[TOP] Building PyG data for: arbiter.v

[TOP] Building PyG data for: vot

In [6]:
# CELL 4 (UPDATED): GCN model + training on M(v)

import torch
from torch import nn
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader

class GCNRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.act = nn.ReLU()
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            h = self.act(h)
        return self.out(h)


# ------------------------------------
# Training loop
# ------------------------------------

if len(data_list) == 0:
    raise RuntimeError("data_list is empty. Check previous cells.")

in_dim = data_list[0].x.size(-1)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"[INFO] Using device: {device}")
print(f"[INFO] Input feature dimension: {in_dim}")

model = GCNRegressor(in_dim=in_dim, hidden_dim=64, num_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

loader = DataLoader(data_list, batch_size=4, shuffle=True)

num_epochs = 50

model.train()
for epoch in range(1, num_epochs + 1):
    total_loss = 0.0
    total_gates = 0

    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index)  # (N, 1)
        gate_mask = batch.gate_mask

        # Only compute loss on gate nodes
        pred_gate = pred[gate_mask]
        y_gate = batch.y[gate_mask]

        if pred_gate.numel() == 0:
            continue

        loss = loss_fn(pred_gate, y_gate)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        num_gates = gate_mask.sum().item()
        total_loss += loss.item() * num_gates
        total_gates += num_gates

    avg_loss = total_loss / max(1, total_gates)
    print(f"[EPOCH {epoch:03d}] MSE (gate nodes): {avg_loss:.6f}")

# ------------------------------------
# Inspect predictions on one circuit
# ------------------------------------

model.eval()
with torch.no_grad():
    d0 = data_list[0].to(device)
    pred = model(d0.x, d0.edge_index).cpu().numpy().flatten()
    gate_mask = d0.gate_mask.cpu().numpy().astype(bool)
    y_true = d0.y.cpu().numpy().flatten()

    print(f"\n[INSPECT] Circuit: {d0.circuit_name}")
    gate_indices = np.where(gate_mask)[0][:10]
    print("First 10 gate-node predictions (pred, true):")
    for idx in gate_indices:
        print(f"  node {idx}: pred={pred[idx]:.3f}, true={y_true[idx]:.3f}")


[INFO] Using device: cpu
[INFO] Input feature dimension: 10
[EPOCH 001] MSE (gate nodes): 0.290750
[EPOCH 002] MSE (gate nodes): 0.072180
[EPOCH 003] MSE (gate nodes): 0.074153
[EPOCH 004] MSE (gate nodes): 0.038612
[EPOCH 005] MSE (gate nodes): 0.030885
[EPOCH 006] MSE (gate nodes): 0.025271
[EPOCH 007] MSE (gate nodes): 0.022529
[EPOCH 008] MSE (gate nodes): 0.020590
[EPOCH 009] MSE (gate nodes): 0.021580
[EPOCH 010] MSE (gate nodes): 0.018131
[EPOCH 011] MSE (gate nodes): 0.017609
[EPOCH 012] MSE (gate nodes): 0.017527
[EPOCH 013] MSE (gate nodes): 0.016839
[EPOCH 014] MSE (gate nodes): 0.015901
[EPOCH 015] MSE (gate nodes): 0.015752
[EPOCH 016] MSE (gate nodes): 0.015703
[EPOCH 017] MSE (gate nodes): 0.016825
[EPOCH 018] MSE (gate nodes): 0.015454
[EPOCH 019] MSE (gate nodes): 0.014390
[EPOCH 020] MSE (gate nodes): 0.014482
[EPOCH 021] MSE (gate nodes): 0.013425
[EPOCH 022] MSE (gate nodes): 0.013042
[EPOCH 023] MSE (gate nodes): 0.014890
[EPOCH 024] MSE (gate nodes): 0.013156
[EPO

In [7]:
# CELL 5: Train/test split by circuit + GCN training + test metrics (MSE/MAE/Spearman)

import random
import numpy as np
import torch
from torch import nn
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader

# ------------------------------------
# GCN model
# ------------------------------------

class GCNRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.act = nn.ReLU()
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            h = self.act(h)
        return self.out(h)


# ------------------------------------
# Spearman correlation (numpy version)
# ------------------------------------

def spearmanr_np(y_true, y_pred):
    """
    Simple Spearman rank correlation implementation using numpy.
    Returns a scalar in [-1, 1] or NaN if undefined.
    """
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    if y_true.size < 2:
        return np.nan

    def rankdata(a):
        order = np.argsort(a)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(a))
        return ranks

    r1 = rankdata(y_true)
    r2 = rankdata(y_pred)

    r1_mean = r1.mean()
    r2_mean = r2.mean()
    num = np.sum((r1 - r1_mean) * (r2 - r2_mean))
    den = np.sqrt(np.sum((r1 - r1_mean) ** 2) * np.sum((r2 - r2_mean) ** 2))
    if den == 0:
        return np.nan
    return num / den


# ------------------------------------
# Train/test split by circuit
# ------------------------------------

if len(data_list) == 0:
    raise RuntimeError("data_list is empty. Make sure Cells 13 ran correctly.")

indices = list(range(len(data_list)))
random.seed(42)
random.shuffle(indices)

split = int(0.8 * len(indices))  # 80% train, 20% test
train_indices = indices[:split]
test_indices = indices[split:]

train_dataset = [data_list[i] for i in train_indices]
test_dataset = [data_list[i] for i in test_indices]

print("[INFO] Train/test split by circuit:")
print("  Train circuits:")
for d in train_dataset:
    print("    ", d.circuit_name)
print("  Test circuits:")
for d in test_dataset:
    print("    ", d.circuit_name)

# ------------------------------------
# Prepare loaders and model
# ------------------------------------

in_dim = data_list[0].x.size(-1)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n[INFO] Using device: {device}")
print(f"[INFO] Input feature dimension: {in_dim}")

model = GCNRegressor(in_dim=in_dim, hidden_dim=64, num_layers=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

num_epochs = 50

# ------------------------------------
# Training loop (train circuits only)
# ------------------------------------

model.train()
for epoch in range(1, num_epochs + 1):
    total_loss = 0.0
    total_gates = 0

    for batch in train_loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index)  # (N, 1)
        gate_mask = batch.gate_mask

        # Only gate nodes contribute to loss
        pred_gate = pred[gate_mask]
        y_gate = batch.y[gate_mask]

        if pred_gate.numel() == 0:
            continue

        loss = loss_fn(pred_gate, y_gate)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        num_gates = gate_mask.sum().item()
        total_loss += loss.item() * num_gates
        total_gates += num_gates

    avg_loss = total_loss / max(1, total_gates)
    print(f"[EPOCH {epoch:03d}] Train MSE (gate nodes): {avg_loss:.6f}")

# ------------------------------------
# Evaluation on test circuits (unseen)
# ------------------------------------

model.eval()
all_true = []
all_pred = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.edge_index)  # (N, 1)
        gate_mask = batch.gate_mask

        pred_gate = pred[gate_mask].cpu().numpy().flatten()
        y_gate = batch.y[gate_mask].cpu().numpy().flatten()

        if pred_gate.size == 0:
            continue

        all_true.append(y_gate)
        all_pred.append(pred_gate)

# Concatenate across all test graphs
if len(all_true) == 0:
    print("\n[WARN] No gate nodes in test set. Check split or data.")
else:
    all_true = np.concatenate(all_true, axis=0)
    all_pred = np.concatenate(all_pred, axis=0)

    mse = np.mean((all_true - all_pred) ** 2)
    mae = np.mean(np.abs(all_true - all_pred))
    spearman = spearmanr_np(all_true, all_pred)

    print("\n[TEST RESULTS] (gate nodes, unseen circuits)")
    print(f"  MSE      : {mse:.6f}")
    print(f"  MAE      : {mae:.6f}")
    print(f"  Spearman : {spearman:.4f}")

    # Optional: quick sanity print for first test circuit
    first_test = test_dataset[0].to(device)
    with torch.no_grad():
        p0 = model(first_test.x, first_test.edge_index).cpu().numpy().flatten()
        gm0 = first_test.gate_mask.cpu().numpy().astype(bool)
        y0 = first_test.y.cpu().numpy().flatten()
        print(f"\n[INSPECT] First test circuit: {first_test.circuit_name}")
        idxs = np.where(gm0)[0][:10]
        print("  First 10 gate-node predictions (pred, true):")
        for idx in idxs:
            print(f"    node {idx}: pred={p0[idx]:.3f}, true={y0[idx]:.3f}")


[INFO] Train/test split by circuit:
  Train circuits:
     bar.v
     c5315.v
     c2670.v
     multiplier.v
     arbiter.v
     router.v
     int2float.v
     cavlc.v
     c3540.v
     i2c.v
     dec.v
     sqrt.v
     div.v
     sin.v
     adder.v
     log2.v
     c432.v
     Priority.v
     c7552.v
     c499.v
     c6288.v
     memctrl.v
     c880.v
     square.v
  Test circuits:
     c1908.v
     c1355.v
     voter.v
     c17.v
     ctrl.v
     max.v

[INFO] Using device: cpu
[INFO] Input feature dimension: 10
[EPOCH 001] Train MSE (gate nodes): 0.361582
[EPOCH 002] Train MSE (gate nodes): 0.123555
[EPOCH 003] Train MSE (gate nodes): 0.060628
[EPOCH 004] Train MSE (gate nodes): 0.049647
[EPOCH 005] Train MSE (gate nodes): 0.038232
[EPOCH 006] Train MSE (gate nodes): 0.027233
[EPOCH 007] Train MSE (gate nodes): 0.021702
[EPOCH 008] Train MSE (gate nodes): 0.018910
[EPOCH 009] Train MSE (gate nodes): 0.016028
[EPOCH 010] Train MSE (gate nodes): 0.014991
[EPOCH 011] Train MSE (gate no

In [8]:
# CELL 6: Multi-seed evaluation, per-circuit Spearman, per-circuit normalization, and baseline

import random
import numpy as np
import torch
from torch import nn
from torch_geometric.nn import GCNConv
from torch_geometric.loader import DataLoader
import networkx as nx

# -------------------------------------------------------------------
# 1) Helper: Spearman correlation (numpy)
# -------------------------------------------------------------------
def spearmanr_np(y_true, y_pred):
    """
    Simple Spearman rank correlation using numpy.
    Returns scalar in [-1, 1] (or NaN if undefined).
    """
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    if y_true.size < 2:
        return np.nan

    def rankdata(a):
        order = np.argsort(a)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(len(a))
        return ranks

    r1 = rankdata(y_true)
    r2 = rankdata(y_pred)

    r1_mean = r1.mean()
    r2_mean = r2.mean()
    num = np.sum((r1 - r1_mean) * (r2 - r2_mean))
    den = np.sqrt(np.sum((r1 - r1_mean) ** 2) * np.sum((r2 - r2_mean) ** 2))
    if den == 0:
        return np.nan
    return num / den


# -------------------------------------------------------------------
# 2) GCN model
# -------------------------------------------------------------------
class GCNRegressor(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.act = nn.ReLU()
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        h = x
        for conv in self.convs:
            h = conv(h, edge_index)
            h = self.act(h)
        return self.out(h)


# -------------------------------------------------------------------
# 3) Per-circuit normalization of M(v): create y_norm, y_mean, y_std
# -------------------------------------------------------------------
print("[INFO] Per-circuit normalization of M(v) (on gate nodes).")

for d in data_list:
    gate_mask = d.gate_mask
    y_gate = d.y[gate_mask]  # (num_gates, 1)

    if y_gate.numel() == 0:
        # No gate nodes; just set norm = y
        d.y_norm = d.y.clone()
        d.y_mean = torch.zeros_like(d.y)
        d.y_std = torch.ones_like(d.y)
        continue

    mean = y_gate.mean().item()
    std = y_gate.std().item()
    if std < 1e-6:
        std = 1.0  # avoid divide-by-zero

    # Store node-level mean/std (same scalar repeated for all nodes in that graph)
    d.y_mean = torch.full_like(d.y, mean)
    d.y_std = torch.full_like(d.y, std)
    d.y_norm = (d.y - mean) / std

print("[INFO] Normalization done for all circuits.")


# -------------------------------------------------------------------
# 4) Baseline: 1 - betweenness centrality on each graph
# -------------------------------------------------------------------
def baseline_betweenness_scores(data):
    """
    Compute baseline score per node for a PyG Data graph:
      baseline(v) = 1 - betweenness_centrality(v)
    Uses an undirected NetworkX graph built from edge_index.
    Returns:
      np.ndarray of shape (num_nodes,)
    """
    num_nodes = data.num_nodes
    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(num_nodes))

    edge_index = data.edge_index.cpu().numpy()
    for u, v in edge_index.T:
        G_nx.add_edge(int(u), int(v))

    if num_nodes < 3:
        bc = {v: 0.0 for v in G_nx.nodes()}
    else:
        try:
            bc = nx.betweenness_centrality(G_nx, normalized=True)
        except ZeroDivisionError:
            bc = {v: 0.0 for v in G_nx.nodes()}

    baseline = np.zeros(num_nodes, dtype=np.float32)
    for v in G_nx.nodes():
        baseline[v] = 1.0 - float(bc.get(v, 0.0))
    return baseline


# -------------------------------------------------------------------
# 5) Multi-seed evaluation
# -------------------------------------------------------------------
if len(data_list) == 0:
    raise RuntimeError("data_list is empty. Make sure Cells 13 ran correctly.")

device = "cuda" if torch.cuda.is_available() else "cpu"
in_dim = data_list[0].x.size(-1)

print(f"[INFO] Using device: {device}")
print(f"[INFO] Input feature dimension: {in_dim}")

seeds = [0, 1, 2, 3, 4]

results_gnn = []
results_base = []

for seed in seeds:
    print(f"\n================ SEED {seed} ================")
    # ---- train/test split by circuit for this seed ----
    indices = list(range(len(data_list)))
    rnd = random.Random(seed)
    rnd.shuffle(indices)

    split = int(0.8 * len(indices))
    train_indices = indices[:split]
    test_indices = indices[split:]

    train_dataset = [data_list[i] for i in train_indices]
    test_dataset = [data_list[i] for i in test_indices]

    print("[SPLIT] Train circuits:", [d.circuit_name for d in train_dataset])
    print("[SPLIT] Test circuits :", [d.circuit_name for d in test_dataset])

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # ---- model & optimizer ----
    model = GCNRegressor(in_dim=in_dim, hidden_dim=64, num_layers=3).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # ---- training (on normalized targets) ----
    num_epochs = 30  # shorter for multiple seeds; adjust if you like
    model.train()
    for epoch in range(1, num_epochs + 1):
        total_loss = 0.0
        total_gates = 0

        for batch in train_loader:
            batch = batch.to(device)
            pred_norm = model(batch.x, batch.edge_index)  # (N, 1)
            gate_mask = batch.gate_mask

            pred_gate_norm = pred_norm[gate_mask]
            y_gate_norm = batch.y_norm[gate_mask]

            if pred_gate_norm.numel() == 0:
                continue

            loss = loss_fn(pred_gate_norm, y_gate_norm)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            num_gates = gate_mask.sum().item()
            total_loss += loss.item() * num_gates
            total_gates += num_gates

        avg_loss = total_loss / max(1, total_gates)
        print(f"[SEED {seed}] Epoch {epoch:03d} | Train MSE (normalized, gate nodes): {avg_loss:.6f}")

    # ---- evaluation on test circuits ----
    model.eval()
    all_true = []
    all_pred = []
    all_base = []

    per_circuit_spearman = []
    per_circuit_base_spearman = []

    with torch.no_grad():
        for data in test_dataset:
            d = data.to(device)
            pred_norm = model(d.x, d.edge_index)  # (N, 1)

            # unnormalize predictions to original M(v) scale
            pred = (pred_norm * d.y_std + d.y_mean).cpu().numpy().flatten()
            y_true = d.y.cpu().numpy().flatten()
            gate_mask = d.gate_mask.cpu().numpy().astype(bool)

            pred_gate = pred[gate_mask]
            y_gate = y_true[gate_mask]

            if pred_gate.size == 0:
                continue

            # collect for global metrics
            all_true.append(y_gate)
            all_pred.append(pred_gate)

            # baseline 1 - betweenness
            base_scores = baseline_betweenness_scores(data)
            base_gate = base_scores[gate_mask]
            all_base.append(base_gate)

            # per-circuit Spearman
            s_gnn = spearmanr_np(y_gate, pred_gate)
            s_base = spearmanr_np(y_gate, base_gate)
            per_circuit_spearman.append((data.circuit_name, s_gnn))
            per_circuit_base_spearman.append((data.circuit_name, s_base))

    if len(all_true) == 0:
        print("[WARN] No gate nodes in test set for this seed.")
        continue

    all_true = np.concatenate(all_true, axis=0)
    all_pred = np.concatenate(all_pred, axis=0)
    all_base = np.concatenate(all_base, axis=0)

    # Global metrics
    mse_gnn = np.mean((all_true - all_pred) ** 2)
    mae_gnn = np.mean(np.abs(all_true - all_pred))
    spearman_gnn = spearmanr_np(all_true, all_pred)

    mse_base = np.mean((all_true - all_base) ** 2)
    mae_base = np.mean(np.abs(all_true - all_base))
    spearman_base = spearmanr_np(all_true, all_base)

    results_gnn.append((mse_gnn, mae_gnn, spearman_gnn))
    results_base.append((mse_base, mae_base, spearman_base))

    print("\n[SEED {} TEST RESULTS] GNN (gate nodes, unseen circuits)".format(seed))
    print(f"  MSE      : {mse_gnn:.6f}")
    print(f"  MAE      : {mae_gnn:.6f}")
    print(f"  Spearman : {spearman_gnn:.4f}")

    print("\n[SEED {} TEST RESULTS] Baseline (1 - betweenness)".format(seed))
    print(f"  MSE      : {mse_base:.6f}")
    print(f"  MAE      : {mae_base:.6f}")
    print(f"  Spearman : {spearman_base:.4f}")

    # Per-circuit Spearman
    print("\n[SEED {}] Per-circuit Spearman (GNN vs baseline)".format(seed))
    for (cname, s_g), (_, s_b) in zip(per_circuit_spearman, per_circuit_base_spearman):
        print(f"  {cname:15s}  GNN={s_g:+.3f}  Base={s_b:+.3f}")

# -------------------------------------------------------------------
# 6) Aggregate results over seeds
# -------------------------------------------------------------------
def summarize_results(results):
    arr = np.array(results)  # shape (seeds, 3)
    mse_mean, mae_mean, sp_mean = arr.mean(axis=0)
    mse_std, mae_std, sp_std = arr.std(axis=0)
    return (mse_mean, mse_std), (mae_mean, mae_std), (sp_mean, sp_std)

if len(results_gnn) > 0:
    (mse_m, mse_s), (mae_m, mae_s), (sp_m, sp_s) = summarize_results(results_gnn)
    print("\n================ AGGREGATE GNN RESULTS OVER SEEDS ================")
    print(f"  MSE      : {mse_m:.6f} ± {mse_s:.6f}")
    print(f"  MAE      : {mae_m:.6f} ± {mae_s:.6f}")
    print(f"  Spearman : {sp_m:.4f} ± {sp_s:.4f}")

if len(results_base) > 0:
    (mse_m, mse_s), (mae_m, mae_s), (sp_m, sp_s) = summarize_results(results_base)
    print("\n================ AGGREGATE BASELINE RESULTS OVER SEEDS ================")
    print(f"  MSE      : {mse_m:.6f} ± {mse_s:.6f}")
    print(f"  MAE      : {mae_m:.6f} ± {mae_s:.6f}")
    print(f"  Spearman : {sp_m:.4f} ± {sp_s:.4f}")


[INFO] Per-circuit normalization of M(v) (on gate nodes).
[INFO] Normalization done for all circuits.
[INFO] Using device: cpu
[INFO] Input feature dimension: 10

================ SEED 0 ================
[SPLIT] Train circuits: ['ctrl.v', 'c2670.v', 'c7552.v', 'c6288.v', 'int2float.v', 'c17.v', 'c1908.v', 'multiplier.v', 'div.v', 'c5315.v', 'max.v', 'sqrt.v', 'sin.v', 'c499.v', 'bar.v', 'c880.v', 'voter.v', 'router.v', 'c3540.v', 'arbiter.v', 'dec.v', 'memctrl.v', 'i2c.v', 'adder.v']
[SPLIT] Test circuits : ['c1355.v', 'c432.v', 'Priority.v', 'square.v', 'cavlc.v', 'log2.v']
[SEED 0] Epoch 001 | Train MSE (normalized, gate nodes): 1.016688
[SEED 0] Epoch 002 | Train MSE (normalized, gate nodes): 1.005036
[SEED 0] Epoch 003 | Train MSE (normalized, gate nodes): 1.002322
[SEED 0] Epoch 004 | Train MSE (normalized, gate nodes): 1.001605
[SEED 0] Epoch 005 | Train MSE (normalized, gate nodes): 0.998621
[SEED 0] Epoch 006 | Train MSE (normalized, gate nodes): 0.996185
[SEED 0] Epoch 007 | T

KeyboardInterrupt: 

In [10]:
# CELL 7: Save preprocessed dataset and gate-type vocabulary for reuse

import torch
import os

save_path = "gnn_structural_netlist_dataset.pt"

# Make sure the key variables exist
print("[CHECK] len(data_list) =", len(data_list))
print("[CHECK] len(all_gate_types) =", len(all_gate_types))

obj_to_save = {
    "data_list": data_list,
    "all_gate_types": all_gate_types,
    "gate_type_to_idx": gate_type_to_idx,
}

torch.save(obj_to_save, save_path)

print(f"[INFO] Saved dataset to: {os.path.abspath(save_path)}")
print("       Contains keys:", list(obj_to_save.keys()))


[CHECK] len(data_list) = 30
[CHECK] len(all_gate_types) = 6
[INFO] Saved dataset to: /data/desktop/rrk307/rupesh_projects/Mutability_analysis/gnn_structural_netlist_dataset.pt
       Contains keys: ['data_list', 'all_gate_types', 'gate_type_to_idx']
